[![Open In Colab](https://colab.research.google.com/assets/colab-badge.png)](https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab3/06_01_autoencoders_mtat_v5.ipynb)

# Lecture 6: Self-supervised learning and autoencoders

## MagnaTagATune — 1-second segments

This notebook trains one deeper autoencoder on MagnaTagATune (MTAT). It uses 30% of the training recordings, the full validation split, and a fixed bottleneck compression rate of 31.25:1.


In [ ]:
# Install deps

!pip install torchcodec
!pip install torch
!pip install mirdata
!pip install tqdm
!pip install scikit-learn
!pip install torchaudio
!pip install matplotlib
!pip install umap-learn
!pip install auraloss torch-audiomentations

In [ ]:
import os
import json
import shutil
import subprocess
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import umap
import auraloss
from torch_audiomentations import Compose, Gain, AddColoredNoise, BandStopFilter, PitchShift, LowPassFilter

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
from IPython.display import Audio

# Set to cuda or cpu, depending on the runtime
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Dataset preparation — MagnaTagATune

The dataset is split by original recording into training and validation subsets (80/20) before sampling. A reproducible 30% subset of the training recordings is used; the validation set remains unchanged. Each sample is one randomly selected 1-second segment from a recording.


In [ ]:
# Mount Google Drive and define the requested results folder
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

RESULTS_DIR = "/content/drive/MyDrive/MLSM/Lab 3/results_v5"
os.makedirs(RESULTS_DIR, exist_ok=True)

# MagnaTagATune files hosted by City, University of London
MTAT_ROOT = "/content/mtat"
AUDIO_DIR = os.path.join(MTAT_ROOT, "mp3")
ANNOTATIONS_PATH = os.path.join(MTAT_ROOT, "annotations_final.csv")

os.makedirs(MTAT_ROOT, exist_ok=True)

# Resume interrupted downloads and verify each file against the server's file size.
import urllib.request

def download_with_resume(url, destination):
    request = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(request, timeout=30) as response:
        expected_size = int(response.headers.get("Content-Length", 0))

    if os.path.exists(destination) and expected_size and os.path.getsize(destination) > expected_size:
        os.remove(destination)

    subprocess.run(["wget", "-c", "--tries=5", "--timeout=30", url, "-P", MTAT_ROOT], check=True)

    downloaded_path = os.path.join(MTAT_ROOT, os.path.basename(url))
    if not os.path.exists(downloaded_path):
        raise FileNotFoundError(f"Download did not create: {downloaded_path}")

    actual_size = os.path.getsize(downloaded_path)
    if expected_size and actual_size != expected_size:
        raise RuntimeError(
            f"Incomplete download for {os.path.basename(url)}: "
            f"{actual_size:,} bytes downloaded, expected {expected_size:,}. "
            "Run this cell again; wget will resume the download."
        )
    if actual_size == 0:
        raise RuntimeError(f"Downloaded file is empty: {downloaded_path}")

    return downloaded_path

# Check annotations on every run; wget -c skips a complete file and resumes a partial one.
download_with_resume(
    "https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv",
    ANNOTATIONS_PATH
)

# The archive contains 16 shards (0-9 and a-f), extracted inside AUDIO_DIR.
expected_shards = set("0123456789abcdef")
os.makedirs(AUDIO_DIR, exist_ok=True)

# Migrate shards if a previous run extracted them one directory too high.
for shard in expected_shards:
    old_location = os.path.join(MTAT_ROOT, shard)
    new_location = os.path.join(AUDIO_DIR, shard)
    if os.path.isdir(old_location) and not os.path.exists(new_location):
        shutil.move(old_location, new_location)

existing_shards = set(os.listdir(AUDIO_DIR))
audio_is_complete = expected_shards.issubset(existing_shards)

if not audio_is_complete:
    archive_parts = ["mp3.zip.001", "mp3.zip.002", "mp3.zip.003"]
    for part in archive_parts:
        download_with_resume(
            f"https://mirg.city.ac.uk/datasets/magnatagatune/{part}",
            os.path.join(MTAT_ROOT, part)
        )

    # Rebuild the joined archive from the verified parts.
    archive_path = os.path.join(MTAT_ROOT, "mp3.zip")
    with open(archive_path, "wb") as output_file:
        for part in archive_parts:
            part_path = os.path.join(MTAT_ROOT, part)
            with open(part_path, "rb") as input_file:
                shutil.copyfileobj(input_file, output_file)

    # The official archive contains folders 0-f at its root: extract them into AUDIO_DIR.
    subprocess.run(["unzip", "-q", "-o", archive_path, "-d", AUDIO_DIR], check=True)

    extracted_shards = set(os.listdir(AUDIO_DIR))
    missing_shards = sorted(expected_shards - extracted_shards)
    if missing_shards:
        raise RuntimeError(
            f"MTAT extraction is incomplete. Missing audio folders: {missing_shards}. "
            f"Top-level entries found: {sorted(extracted_shards)[:30]}"
        )

print("Audio directory:", AUDIO_DIR)
print("Annotations file:", ANNOTATIONS_PATH)
print("Audio directory exists:", os.path.isdir(AUDIO_DIR))


In [ ]:
# Read MTAT annotations
annotations = pd.read_csv(ANNOTATIONS_PATH, sep="\t")

# Exclude three empty MP3 files
invalid_files = {
    "6/norine_braun-now_and_zen-08-gently-117-146.mp3",
    "8/jacob_heringman-josquin_des_prez_lute_settings-19-gintzler__pater_noster-204-233.mp3",
    "9/american_baroque-dances_and_suites_of_rameau_and_couperin-25-le_petit_rien_xiveme_ordre_couperin-88-117.mp3",
}

annotations = annotations[
    ~annotations["mp3_path"].isin(invalid_files)
].copy()

# Identify metadata columns and tag columns
metadata_columns = [column for column in ["clip_id", "mp3_path"] if column in annotations.columns]
tag_columns = [column for column in annotations.columns if column not in metadata_columns]

# Keep rows with an available audio path and at least one positive tag
annotations = annotations.dropna(subset=["mp3_path"]).copy()
annotations[tag_columns] = annotations[tag_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
annotations = annotations.loc[annotations[tag_columns].sum(axis=1) > 0].reset_index(drop=True)

# Use the most strongly annotated tag as the display label for UMAP
annotations["natural_label"] = annotations[tag_columns].idxmax(axis=1)

# Split by original recording, not by segment, to avoid data leakage
train_rows, val_rows = train_test_split(
    np.arange(len(annotations)),
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_annotations = annotations.iloc[train_rows].reset_index(drop=True)
val_annotations = annotations.iloc[val_rows].reset_index(drop=True)

# Use the same 30% training-data subsampling criterion as the Lab 4 reference.
# Keep the validation set unchanged and sample reproducibly.
full_train_size = len(train_annotations)
train_annotations = train_annotations.sample(
    n=int(full_train_size * 0.3),
    random_state=42
).reset_index(drop=True)

print(f"Tracks: {len(annotations)}")
print(f"Training tracks before subsampling: {full_train_size}")
print(f"Training tracks used (30%): {len(train_annotations)}")
print(f"Validation tracks (100%): {len(val_annotations)}")
print(f"Number of tags: {len(tag_columns)}")


In [ ]:
class MTATSegmentDataset(Dataset):
    """MTAT dataset returning one random 1-second segment per recording."""

    def __init__(
        self,
        annotations,
        audio_root,
        sample_rate=16000,
        segment_duration=1
    ):
        self.annotations = annotations.reset_index(drop=True)
        self.audio_root = audio_root
        self.sample_rate = sample_rate
        self.segment_duration = segment_duration
        self.segment_samples = sample_rate * segment_duration

    def __len__(self):
        # One sample per recording in each epoch.
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        audio_path = os.path.join(self.audio_root, str(row["mp3_path"]))

        # Load and decode the MP3 every time a sample is requested.
        audio, sr = torchaudio.load(audio_path)

        # Convert to mono.
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)

        # Resample to 16 kHz when necessary.
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
            audio = resampler(audio)

        audio = audio.squeeze(0)

        # Normalize the complete recording.
        max_val = torch.abs(audio).max()
        if max_val > 0:
            audio = audio / max_val

        # Select one random 1-second segment from the first 27 seconds.
        max_start_second = max(
            0,
            int(np.ceil(len(audio) / self.sample_rate)) - self.segment_duration
        )
        max_start_second = min(max_start_second, 26)
        start_second = torch.randint(0, max_start_second + 1, (1,)).item()

        start = start_second * self.sample_rate
        end = start + self.segment_samples
        segment = audio[start:end]

        if len(segment) < self.segment_samples:
            segment = F.pad(segment, (0, self.segment_samples - len(segment)))

        return {
            "audio": segment,
            "natural_labels": row["natural_label"],
            "track_id": str(row["clip_id"]) if "clip_id" in row else str(idx)
        }


train_dataset = MTATSegmentDataset(train_annotations, AUDIO_DIR)
val_dataset = MTATSegmentDataset(val_annotations, AUDIO_DIR)

train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True, num_workers=2,
    pin_memory=(device == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=128, shuffle=False, num_workers=2,
    pin_memory=(device == "cuda")
)

print(f"Training recordings: {len(train_dataset)}")
print(f"Validation recordings: {len(val_dataset)}")
print(f"Segment length: {train_dataset.segment_samples} samples ({train_dataset.segment_duration} second)")
print("Audio storage: loaded on demand; one random segment per recording; no preload or cache")


# Model and training strategy

A single improved autoencoder is trained. It keeps the two additional encoder/decoder blocks and the original bottleneck compression rate (31.25:1). Instead of running separate architecture and STFT experiments, the model is optimized with a weighted combination of waveform MSE and a perceptually weighted multi-resolution STFT loss, following the combined-loss strategy in the Lab 4 reference notebook.

The training objective is \(L = 0.1 L_{MSE} + 1.0 L_{perceptual}\). Data augmentation is applied during training only; validation uses the unmodified audio.

In [ ]:
# Keep the original compression rate: 16,000 / 512 = 31.25
target_length = 16000
bottleneck_size = 512
downsample = 2
num_epochs = 20
COMPRESSION_RATE = target_length / bottleneck_size
assert abs(COMPRESSION_RATE - 31.25) < 1e-8
print(f"Compression rate: {COMPRESSION_RATE:.2f}:1")

class AugmentationChain(nn.Module):
    """Audio augmentation pipeline adapted from the Lab 4 reference notebook."""
    def __init__(self):
        super().__init__()
        self.chain = Compose(
            transforms=[
                Gain(p=0.75, output_type="tensor"),
                BandStopFilter(sample_rate=16000, p=0.75, output_type="tensor"),
                PitchShift(p=0.75, sample_rate=16000, output_type="tensor"),
                AddColoredNoise(p=0.75, output_type="tensor"),
                LowPassFilter(
                    p=0.75, sample_rate=16000,
                    min_cutoff_freq=400, output_type="tensor"
                ),
            ],
            output_type="tensor"
        )

    def forward(self, audio):
        return self.chain(audio.unsqueeze(1)).squeeze(1)


def train_improved_autoencoder(
    model,
    device,
    train_loader,
    val_loader,
    num_epochs=20,
    learning_rate=1e-3,
    use_augmentation=True,
    alpha=0.1,
    beta=1.0,
    save_best_model=True,
    checkpoint_path=None
):
    """Train with a weighted combination of waveform MSE and auraloss MR-STFT."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    mse_criterion = nn.MSELoss()

    # Same perceptual-loss family and FFT resolutions as the Lab 4 reference.
    perceptual_criterion = auraloss.freq.MultiResolutionSTFTLoss(
        fft_sizes=[512, 1024, 2048],
        hop_sizes=[128, 256, 512],
        win_lengths=[512, 1024, 2048],
        scale="mel",
        n_bins=128,
        sample_rate=16000,
        perceptual_weighting=True,
    ).to(device)

    augmentation = AugmentationChain().to(device) if use_augmentation else None
    if augmentation is not None:
        augmentation.eval()
        print("Data augmentation enabled")

    history = {
        "train_loss": [], "val_loss": [],
        "train_mse": [], "train_perceptual": [],
        "val_mse": [], "val_perceptual": []
    }
    best_val_mse = float("inf")
    best_epoch = 0

    for epoch in range(num_epochs):
        model.train()
        train_losses, train_mse_losses, train_perceptual_losses = [], [], []
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

        for batch in train_pbar:
            audio = batch["audio"].to(device)
            if augmentation is not None:
                with torch.no_grad():
                    audio = augmentation(audio)

            optimizer.zero_grad()
            recon_audio, _ = model(audio.unsqueeze(1))
            recon_audio = recon_audio.squeeze(1)

            mse_value = mse_criterion(recon_audio, audio)
            perceptual_value = perceptual_criterion(
                recon_audio.unsqueeze(1), audio.unsqueeze(1)
            )
            loss = alpha * mse_value + beta * perceptual_value
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_mse_losses.append(mse_value.item())
            train_perceptual_losses.append(perceptual_value.item())
            train_pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "mse": f"{mse_value.item():.4f}",
                "perc": f"{perceptual_value.item():.4f}"
            })

        model.eval()
        val_losses, val_mse_losses, val_perceptual_losses = [], [], []
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            for batch in val_pbar:
                audio = batch["audio"].to(device)
                recon_audio, _ = model(audio.unsqueeze(1))
                recon_audio = recon_audio.squeeze(1)

                mse_value = mse_criterion(recon_audio, audio)
                perceptual_value = perceptual_criterion(
                    recon_audio.unsqueeze(1), audio.unsqueeze(1)
                )
                val_loss = alpha * mse_value + beta * perceptual_value

                val_losses.append(val_loss.item())
                val_mse_losses.append(mse_value.item())
                val_perceptual_losses.append(perceptual_value.item())
                val_pbar.set_postfix({
                    "loss": f"{val_loss.item():.4f}",
                    "mse": f"{mse_value.item():.4f}",
                    "perc": f"{perceptual_value.item():.4f}"
                })

        avg_train_loss = float(np.mean(train_losses))
        avg_val_loss = float(np.mean(val_losses))
        history["train_loss"].append(avg_train_loss)
        history["train_mse"].append(float(np.mean(train_mse_losses)))
        history["train_perceptual"].append(float(np.mean(train_perceptual_losses)))
        history["val_loss"].append(avg_val_loss)
        history["val_mse"].append(float(np.mean(val_mse_losses)))
        history["val_perceptual"].append(float(np.mean(val_perceptual_losses)))

        print(
            f"Epoch {epoch+1}: train={avg_train_loss:.6f}, "
            f"val={avg_val_loss:.6f}, val_MSE={history['val_mse'][-1]:.6f}, "
            f"val_perceptual={history['val_perceptual'][-1]:.6f}"
        )

        # Select the best checkpoint exclusively by the lowest validation MSE.
        # The combined MSE + perceptual loss remains the optimization objective.
        avg_val_mse = history["val_mse"][-1]
        if save_best_model and avg_val_mse < best_val_mse:
            best_val_mse = avg_val_mse
            best_epoch = epoch + 1
            if checkpoint_path is not None:
                torch.save({
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": avg_val_loss,
                    "val_mse": avg_val_mse,
                    "history": history,
                    "config": {
                        "alpha": alpha, "beta": beta,
                        "selection_metric": "validation_mse",
                        "use_augmentation": use_augmentation,
                        "learning_rate": learning_rate
                    }
                }, checkpoint_path)
                print(
                    f"Saved best checkpoint at epoch {epoch+1} "
                    f"(validation MSE: {avg_val_mse:.6f})"
                )

    if save_best_model and checkpoint_path is not None and os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(
            f"Restored best model from epoch {checkpoint['epoch']} "
            f"(validation MSE: {checkpoint['val_mse']:.6f})"
        )

    with open(os.path.join(RESULTS_DIR, "combined_training_history.json"), "w") as file:
        json.dump(history, file, indent=2)

    return history

# Improved autoencoder — additional convolutional layers

The improved model adds two convolutional/pooling blocks to the encoder and mirrored decoder. It is trained once with the combined MSE and perceptual objective, rather than running separate architecture and STFT experiments.

In [ ]:
class AudioAutoencoderImproved(nn.Module):
    """Deeper version of the original autoencoder with mirrored encoder/decoder blocks."""

    def __init__(
        self,
        input_length=16000,
        bottleneck_size=512,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=2,
        extra_layers=1,
    ):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd for same padding"

        self.num_layers = 3 + extra_layers
        self.downsample = downsample
        padding = kernel_size // 2

        self.encoder_convs = nn.ModuleList()
        self.encoder_norms = nn.ModuleList()
        self.encoder_acts = nn.ModuleList()
        self.pools = nn.ModuleList()

        self.channel_sizes = [channels * (channel_mult ** i) for i in range(self.num_layers)]

        in_channels = 1
        for out_channels in self.channel_sizes:
            self.encoder_convs.append(
                nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding)
            )
            self.encoder_norms.append(nn.BatchNorm1d(out_channels))
            self.encoder_acts.append(nn.PReLU())
            self.pools.append(nn.MaxPool1d(2, stride=downsample, return_indices=True))
            in_channels = out_channels

        enc_out_len = (input_length // downsample ** self.num_layers) * self.channel_sizes[-1]
        self.enc_fc = nn.Linear(enc_out_len, bottleneck_size)
        self.dec_fc = nn.Linear(bottleneck_size, enc_out_len)

        self.unpools = nn.ModuleList()
        self.decoder_convs = nn.ModuleList()
        self.decoder_acts = nn.ModuleList()

        for layer_idx in reversed(range(self.num_layers)):
            in_ch = self.channel_sizes[layer_idx]
            out_ch = self.channel_sizes[layer_idx - 1] if layer_idx > 0 else 1
            self.unpools.append(nn.MaxUnpool1d(2, stride=downsample))
            self.decoder_convs.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=padding)
            )
            self.decoder_acts.append(nn.PReLU() if layer_idx > 0 else nn.Tanh())

        comp_rate = input_length / bottleneck_size
        print(f"Improved compression rate: {comp_rate:.2f}:1 | Extra layers: {extra_layers}")

    def encoder(self, x):
        sizes, indices = [], []

        for conv, norm, act, pool in zip(
            self.encoder_convs, self.encoder_norms, self.encoder_acts, self.pools
        ):
            x = act(norm(conv(x)))
            sizes.append(x.size())
            x, idx = pool(x)
            indices.append(idx)

        x = x.flatten(start_dim=1)
        z = self.enc_fc(x)
        return z, indices, sizes

    def decoder(self, z, indices, sizes):
        x = self.dec_fc(z)
        x = x.view(sizes[-1][0], sizes[-1][1], -1)

        for idx, size, unpool, conv, act in zip(
            reversed(indices), reversed(sizes), self.unpools,
            self.decoder_convs, self.decoder_acts
        ):
            x = unpool(x, idx, output_size=size)
            x = act(conv(x))

        return x

    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        x_reconstructed = self.decoder(z, indices, sizes)
        return x_reconstructed, z

In [ ]:
# Train the single improved model with combined MSE + perceptual loss.
improved_results = {}
extra_layers = 2
model_name = f"improved_{extra_layers}_extra_layers_combined"

improved_model = AudioAutoencoderImproved(
    input_length=target_length,
    bottleneck_size=bottleneck_size,
    downsample=downsample,
    extra_layers=extra_layers
)

checkpoint_path = os.path.join(RESULTS_DIR, f"{model_name}_best.pth")
improved_history = train_improved_autoencoder(
    improved_model,
    device,
    train_loader,
    val_loader,
    num_epochs=num_epochs,
    learning_rate=1e-3,
    use_augmentation=True,
    alpha=0.1,
    beta=1.0,
    save_best_model=True,
    checkpoint_path=checkpoint_path
)

improved_results[model_name] = {
    "model": improved_model,
    "history": improved_history,
    "config": {
        "extra_layers": extra_layers,
        "loss": "0.1 * MSE + 1.0 * perceptual MR-STFT",
        "alpha": 0.1,
        "beta": 1.0,
        "augmentation": True
    }
}

# Training loss evolution: MSE and STFT

The results show the epoch-by-epoch evolution of both loss components used in training. The first plot compares training and validation waveform MSE; the second compares training and validation multi-resolution STFT perceptual loss. Both metrics are reported separately so that their behaviour can be assessed throughout the 20 epochs.

In [ ]:
result = improved_results[model_name]
history = result["history"]

epochs = range(1, len(history["train_mse"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Waveform-domain reconstruction loss
axes[0].plot(epochs, history["train_mse"], marker="o", markersize=4, label="Train MSE")
axes[0].plot(epochs, history["val_mse"], marker="o", markersize=4, label="Validation MSE")
axes[0].set_title("MSE loss per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE loss")
axes[0].set_xticks(list(epochs))
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Multi-resolution STFT perceptual reconstruction loss
axes[1].plot(epochs, history["train_perceptual"], marker="o", markersize=4, label="Train STFT")
axes[1].plot(epochs, history["val_perceptual"], marker="o", markersize=4, label="Validation STFT")
axes[1].set_title("Multi-resolution STFT loss per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("STFT perceptual loss")
axes[1].set_xticks(list(epochs))
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "mse_stft_train_validation_curves.png"), dpi=200)
plt.show()

# UMAP projection of the latent representations

The UMAP projection is generated for the single trained combined-loss model using validation embeddings.

In [ ]:
all_results = improved_results

In [ ]:
def plot_umap(model, model_name, max_points=5000):
    model.eval()
    embeddings = []
    natural_labels = []
    collected = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Extracting embeddings: {model_name}"):
            x = batch["audio"].unsqueeze(1).to(device)
            _, z = model(x)

            embeddings.append(z.cpu().numpy())
            natural_labels.extend(batch["natural_labels"])
            collected += len(z)

            if collected >= max_points:
                break

    embeddings = np.concatenate(embeddings, axis=0)[:max_points]
    natural_labels = np.asarray(natural_labels)[:max_points]

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42
    )
    umap_embeddings = reducer.fit_transform(embeddings)

    unique_labels, label_indices = np.unique(natural_labels, return_inverse=True)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        umap_embeddings[:, 0],
        umap_embeddings[:, 1],
        c=label_indices,
        cmap="tab20",
        s=6,
        alpha=0.75
    )

    handles, _ = scatter.legend_elements(num=min(len(unique_labels), 20))
    legend_labels = unique_labels[:len(handles)]
    plt.legend(handles, legend_labels, title="Most common MTAT tag", loc="best", fontsize=7)

    plt.title(f"UMAP of Audio Embeddings — {model_name}")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"umap_{model_name}.png"), dpi=200)
    plt.show()


for model_name, result in all_results.items():
    plot_umap(result["model"], model_name)

# Final metrics

In [ ]:
# Save the final validation metrics and experiment settings.
summary = []
for model_name, result in all_results.items():
    history = result["history"]
    config = result.get("config", {})
    summary.append({
        "model": model_name,
        "final_train_mse": history["train_mse"][-1],
        "final_validation_mse": history["val_mse"][-1],
        "final_train_perceptual": history["train_perceptual"][-1],
        "final_validation_perceptual": history["val_perceptual"][-1],
        "final_train_combined_loss": history["train_loss"][-1],
        "final_validation_combined_loss": history["val_loss"][-1],
        "alpha_mse": config.get("alpha", 0.1),
        "beta_perceptual": config.get("beta", 1.0),
        "compression_rate": COMPRESSION_RATE,
        "segment_duration_seconds": 1,
        "sample_rate": 16000,
        "epochs": num_epochs
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(os.path.join(RESULTS_DIR, "training_summary.csv"), index=False)
summary_df

# 7. Listen to the original audio and reconstructions

All trained models are evaluated on the same five consecutive 1-second segments from one validation recording. Each segment is reconstructed independently, and the five outputs are concatenated for comparison.

In [ ]:
from IPython.display import Audio, display

# Select one validation recording and apply the same audio preprocessing.
reference_row = val_annotations.iloc[0]
reference_path = os.path.join(AUDIO_DIR, str(reference_row["mp3_path"]))
audio, sr = torchaudio.load(reference_path)

if audio.shape[0] > 1:
    audio = audio.mean(dim=0, keepdim=True)
if sr != 16000:
    audio = torchaudio.transforms.Resample(sr, 16000)(audio)

audio = audio.squeeze(0)
peak = audio.abs().max()
if peak > 0:
    audio = audio / peak

# Select five consecutive seconds within the first 27 seconds.
segment_samples = 16000
max_start = min(max(0, len(audio) // segment_samples - 5), 22)
start_second = int(torch.randint(0, max_start + 1, (1,)).item())
start = start_second * segment_samples
original = audio[start:start + 5 * segment_samples]

if len(original) < 5 * segment_samples:
    original = F.pad(original, (0, 5 * segment_samples - len(original)))

original_segments = original.split(segment_samples)
print("Reference track:", reference_row["clip_id"])
print("MTAT tag:", reference_row["natural_label"])
print("Start time:", start_second, "seconds")
print("\nORIGINAL AUDIO — 5 SECONDS")
display(Audio(original.cpu().numpy(), rate=16000))

# Reconstruct the same five segments with the single combined-loss model.
for model_name, result in all_results.items():
    model = result["model"]
    model.eval()
    reconstructed_segments = []

    with torch.no_grad():
        for segment in original_segments:
            model_input = segment.unsqueeze(0).unsqueeze(0).to(device)
            reconstruction, _ = model(model_input)
            reconstructed_segments.append(reconstruction.squeeze().cpu())

    reconstructed_audio = torch.cat(reconstructed_segments)

    print(f"\nRECONSTRUCTION — {model_name} — 5 SECONDS")
    display(Audio(reconstructed_audio.numpy(), rate=16000))

    torchaudio.save(
        os.path.join(RESULTS_DIR, f"audio_reference_5segments_{model_name}.wav"),
        reconstructed_audio.unsqueeze(0),
        sample_rate=16000
    )

torchaudio.save(
    os.path.join(RESULTS_DIR, "audio_reference_5segments_original.wav"),
    original.cpu().unsqueeze(0),
    sample_rate=16000
)

# Report

**Data Loading and Preprocessing**

The MagnaTagATune (MTAT) dataset is processed using on-demand loading. Each recording is decoded when requested by the DataLoader. The recordings are split at track level into training and validation subsets using an 80/20 split and a fixed random seed. A reproducible 30% subset of the training recordings is used, while the full validation subset is retained. Each recording is converted to mono, resampled to 16 kHz when necessary, normalized, and cropped to one randomly selected second from the first 27 seconds. The selected segment may vary between epochs. The DataLoaders use a batch size of 128 and two worker processes. No audio preloading or caching is used.

**Single Improved Autoencoder with Combined Loss**

Only one model is trained: the improved autoencoder with two additional encoder/decoder blocks. The bottleneck size is 512 for 16,000 input samples, preserving the compression rate of 31.25:1. The model is trained for 20 epochs using Adam with a learning rate of 0.001. Its objective combines waveform MSE and a perceptually weighted multi-resolution STFT loss from auraloss:

\[
L = 0.1 L_{MSE} + 1.0 L_{perceptual}.
\]

The perceptual loss uses mel scaling, perceptual weighting, and FFT sizes of 512, 1024, and 2048 samples. Data augmentation (gain, band-stop filtering, pitch shifting, colored noise, and low-pass filtering) is applied during training only. Validation is performed on unaugmented audio, and the best checkpoint is selected using the lowest validation MSE. The combined objective is used for gradient optimization, not for checkpoint selection.

**Evaluation**

The model is evaluated using validation MSE and perceptual loss, latent-space UMAP visualization, and listening comparisons. The same five consecutive one-second segments from one validation recording are used for the original audio and its reconstruction. This notebook therefore contains one training experiment that combines the deeper architecture and perceptual-loss strategy, rather than separate deeper-model and STFT-model experiments.